# Human-Only Analysis by Question Type

Focused analysis of human answer patterns (entropy, accuracy, agreement, confidence)
broken down by **operation type** and entity group. Key finding: operation type
significantly differentiates human behavior while entity group does not.

In [ ]:
import sys, numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
from scipy.stats import entropy as sp_entropy
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

ROOT = Path('.').resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'analysis'))
from config import MODELS_7B
from utils.constants import VARIANT_ORDER

EXPORTS = ROOT / 'analysis/session2/exports'
OUT_DIR = Path('.').resolve()
_7b = set(MODELS_7B)

human = pd.read_csv(EXPORTS / 'responses_human.csv')
pc = pd.read_parquet(EXPORTS / 'pair_cache_cleaned.parquet')

# Question metadata from variant C
meta = human[human['variant'] == 'C'].drop_duplicates('question_id')[
    ['question_id', 'question_en', 'ent', 'op', 'gt']
].set_index('question_id')

ENT_MERGE = {
    'person': 'person', 'animal': 'animal', 'object': 'object', 'food': 'food',
    'other': 'other', 'product': 'other', 'place': 'other',
    'vehicle': 'other', 'text': 'other',
}
meta['ent_group'] = meta['ent'].map(ENT_MERGE).fillna('other')

OP_ORDER = ['ident', 'count', 'attr', 'act', 'spat']
OP_LABELS = {'ident': 'Identity', 'count': 'Count', 'attr': 'Attribute',
             'act': 'Action', 'spat': 'Spatial'}
ENT_ORDER = ['person', 'animal', 'object', 'food', 'other']

print(f'Human responses: {len(human):,}')
print(f'Questions: {len(meta)}')
print(f'Participants: {human["participant"].nunique()}')
print(f'Op distribution: {meta["op"].value_counts().to_dict()}')
print(f'Entity distribution: {meta["ent_group"].value_counts().to_dict()}')

## 1. Per-Question Human Metrics

For each question and **each variant** (C/B/A): entropy, accuracy, HH SBERT agreement.

In [ ]:
# Compute per-question metrics for ALL variants
def answer_entropy(grp):
    counts = grp['response'].value_counts()
    probs = counts / counts.sum()
    return sp_entropy(probs, base=2)

qdf_all = {}  # variant -> qdf
for v in VARIANT_ORDER:
    hv = human[human['variant'] == v]

    h_entropy = hv.groupby('question_id').apply(
        answer_entropy, include_groups=False
    ).rename('entropy')

    h_acc = hv.groupby('question_id')['accuracy'].mean().rename('accuracy')

    hh_sbert = pc[(pc['variant'] == v) & (pc['pair_type'] == 'HH')].groupby(
        'question_id'
    )['sbert_score'].mean().rename('hh_sbert')

    h_nunique = hv.groupby('question_id')['response'].nunique().rename('n_unique')

    df = pd.DataFrame({
        'entropy': h_entropy, 'accuracy': h_acc,
        'hh_sbert': hh_sbert, 'n_unique': h_nunique,
    }).join(meta)
    df['variant'] = v
    qdf_all[v] = df

# Primary reference: variant C
hC = human[human['variant'] == 'C']
qdf = qdf_all['C']

print(f'Questions with all metrics (per variant):')
for v in VARIANT_ORDER:
    print(f'  {v}: {len(qdf_all[v].dropna())}')
print(f'\nGlobal stats (variant C):')
print(qdf[['entropy', 'accuracy', 'hh_sbert', 'n_unique']].describe().round(3))

## 2. Human Patterns by Operation Type

In [ ]:
# Table: human metrics by op × variant
op_rows = []
for op in OP_ORDER:
    row = {'Operation': OP_LABELS[op], 'N': len(qdf[qdf['op'] == op])}
    for v in VARIANT_ORDER:
        sub = qdf_all[v][qdf_all[v]['op'] == op]
        row[f'Entropy {v}'] = sub['entropy'].mean()
        row[f'Acc {v}'] = sub['accuracy'].mean()
        row[f'HH SBERT {v}'] = sub['hh_sbert'].mean()
    op_rows.append(row)

# All row
row_all = {'Operation': 'All', 'N': len(qdf)}
for v in VARIANT_ORDER:
    row_all[f'Entropy {v}'] = qdf_all[v]['entropy'].mean()
    row_all[f'Acc {v}'] = qdf_all[v]['accuracy'].mean()
    row_all[f'HH SBERT {v}'] = qdf_all[v]['hh_sbert'].mean()
op_rows.append(row_all)

op_tbl = pd.DataFrame(op_rows).set_index('Operation')
fmt = {'N': '{:d}'}
for v in VARIANT_ORDER:
    fmt[f'Entropy {v}'] = '{:.2f}'
    fmt[f'Acc {v}'] = '{:.3f}'
    fmt[f'HH SBERT {v}'] = '{:.3f}'
display(op_tbl.style.format(fmt))

# ANOVA on variant C
groups = [qdf[qdf['op'] == op]['entropy'].values for op in OP_ORDER if len(qdf[qdf['op'] == op]) > 1]
F, p = stats.f_oneway(*groups)
print(f'\nANOVA entropy ~ op (variant C): F={F:.2f}, p={p:.4f}')

# Check ANOVA holds across variants
for v in VARIANT_ORDER:
    vdf = qdf_all[v]
    groups_v = [vdf[vdf['op'] == op]['entropy'].values for op in OP_ORDER if len(vdf[vdf['op'] == op]) > 1]
    Fv, pv = stats.f_oneway(*groups_v)
    print(f'  Variant {v}: F={Fv:.2f}, p={pv:.4f}')

In [ ]:
# Bootstrap CIs and permutation test for robustness with unequal N
rng = np.random.default_rng(42)
N_BOOT = 5000
N_PERM = 10000

# 1. Bootstrap 95% CI for each group mean
print('=== Bootstrap 95% CI for mean entropy by op (5000 resamples) ===')
boot_ci = {}
for op in OP_ORDER:
    vals = qdf[qdf['op'] == op]['entropy'].values
    n = len(vals)
    boot_means = [np.mean(rng.choice(vals, size=n, replace=True)) for _ in range(N_BOOT)]
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    boot_ci[op] = (lo, hi)
    print('  {:10s}: mean={:.2f}, 95% CI=[{:.2f}, {:.2f}], N={}'.format(
        OP_LABELS[op], np.mean(vals), lo, hi, n))

# 2. Permutation ANOVA
obs_F, _ = stats.f_oneway(*[qdf[qdf['op'] == op]['entropy'].values for op in OP_ORDER])
all_entropy = qdf['entropy'].values
all_op = qdf['op'].values
perm_count = 0
for _ in range(N_PERM):
    perm_op = rng.permutation(all_op)
    groups_perm = [all_entropy[perm_op == op] for op in OP_ORDER if (perm_op == op).sum() > 0]
    if len(groups_perm) >= 2:
        F_perm, _ = stats.f_oneway(*groups_perm)
        if F_perm >= obs_F:
            perm_count += 1
p_perm = perm_count / N_PERM

# Kruskal-Wallis (non-parametric)
H_stat, p_kw = stats.kruskal(*[qdf[qdf['op'] == op]['entropy'].values for op in OP_ORDER])

print('\n=== Non-parametric tests ===')
print('  Permutation ANOVA: F={:.2f}, p={:.4f}'.format(obs_F, p_perm))
print('  Kruskal-Wallis:    H={:.2f}, p={:.4f}'.format(H_stat, p_kw))

# 3. Pairwise bootstrap differences
print('\n=== Pairwise bootstrap: act vs others ===')
act_vals = qdf[qdf['op'] == 'act']['entropy'].values
for op in ['ident', 'count', 'attr', 'spat']:
    other_vals = qdf[qdf['op'] == op]['entropy'].values
    obs_diff = np.mean(act_vals) - np.mean(other_vals)
    boot_diffs = [
        np.mean(rng.choice(act_vals, size=len(act_vals), replace=True)) -
        np.mean(rng.choice(other_vals, size=len(other_vals), replace=True))
        for _ in range(N_BOOT)
    ]
    lo, hi = np.percentile(boot_diffs, [2.5, 97.5])
    sig = 'Yes' if lo > 0 or hi < 0 else 'No'
    print('  act - {:10s}: diff={:+.2f}, 95% CI=[{:+.2f}, {:+.2f}], sig={}'.format(
        OP_LABELS[op], obs_diff, lo, hi, sig))

# 4. Effect size
SS_between = sum(
    len(qdf[qdf['op'] == op]) * (qdf[qdf['op'] == op]['entropy'].mean() - qdf['entropy'].mean())**2
    for op in OP_ORDER
)
SS_total = ((qdf['entropy'] - qdf['entropy'].mean())**2).sum()
eta_sq = SS_between / SS_total
print('\nEffect size: eta^2 = {:.3f}'.format(eta_sq))

## 3. Human Patterns by Entity Group

In [ ]:
# Table: human metrics by entity × variant
ent_rows = []
for eg in ENT_ORDER:
    row = {'Entity': eg.capitalize(), 'N': len(qdf[qdf['ent_group'] == eg])}
    for v in VARIANT_ORDER:
        sub = qdf_all[v][qdf_all[v]['ent_group'] == eg]
        row[f'Entropy {v}'] = sub['entropy'].mean()
        row[f'Acc {v}'] = sub['accuracy'].mean()
        row[f'HH SBERT {v}'] = sub['hh_sbert'].mean()
    ent_rows.append(row)

row_all = {'Entity': 'All', 'N': len(qdf)}
for v in VARIANT_ORDER:
    row_all[f'Entropy {v}'] = qdf_all[v]['entropy'].mean()
    row_all[f'Acc {v}'] = qdf_all[v]['accuracy'].mean()
    row_all[f'HH SBERT {v}'] = qdf_all[v]['hh_sbert'].mean()
ent_rows.append(row_all)

ent_tbl = pd.DataFrame(ent_rows).set_index('Entity')
fmt = {'N': '{:d}'}
for v in VARIANT_ORDER:
    fmt[f'Entropy {v}'] = '{:.2f}'
    fmt[f'Acc {v}'] = '{:.3f}'
    fmt[f'HH SBERT {v}'] = '{:.3f}'
display(ent_tbl.style.format(fmt))

# ANOVA across variants
for v in VARIANT_ORDER:
    vdf = qdf_all[v]
    groups_ent = [vdf[vdf['ent_group'] == eg]['entropy'].values for eg in ENT_ORDER]
    Fv, pv = stats.f_oneway(*groups_ent)
    print(f'ANOVA entropy ~ entity (variant {v}): F={Fv:.2f}, p={pv:.4f}')

## 4. Visualisation: Entropy by Operation Type

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
variant_colors = {'C': '#1f77b4', 'B': '#ff7f0e', 'A': '#2ca02c'}

for vi, v in enumerate(VARIANT_ORDER):
    ax = axes[vi]
    vdf = qdf_all[v]
    data_op = [vdf[vdf['op'] == op]['entropy'].values for op in OP_ORDER]
    bp = ax.boxplot(data_op, labels=[OP_LABELS[op] for op in OP_ORDER],
                    patch_artist=True, widths=0.6)
    for patch in bp['boxes']:
        patch.set_facecolor(variant_colors[v])
        patch.set_alpha(0.6)
    ax.set_ylabel('Answer entropy (bits)' if vi == 0 else '')
    ax.set_title(f'Variant {v}')
    ax.set_ylim(-0.2, 6.0)
    for i, op in enumerate(OP_ORDER):
        n = len(vdf[vdf['op'] == op])
        ax.text(i + 1, -0.1, f'n={n}', ha='center', fontsize=8, color='gray')

fig.suptitle('Human answer entropy by operation type × variant', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'human_entropy_by_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: human_entropy_by_type.png')

## 5. Entropy–Accuracy Relationship by Op

Do high-entropy (diverse) questions also have low accuracy? This tests whether
human diversity reflects genuine ambiguity vs random guessing.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
op_colors = plt.cm.Set2(np.linspace(0, 1, len(OP_ORDER)))

for vi, v in enumerate(VARIANT_ORDER):
    ax = axes[vi]
    vdf = qdf_all[v]
    for i, op in enumerate(OP_ORDER):
        sub = vdf[vdf['op'] == op]
        ax.scatter(sub['entropy'], sub['accuracy'],
                   label=f'{OP_LABELS[op]} (n={len(sub)})',
                   alpha=0.7, s=40, color=op_colors[i])

    valid = vdf[['entropy', 'accuracy']].dropna()
    r, p = stats.pearsonr(valid['entropy'], valid['accuracy'])
    ax.set_xlabel('Human answer entropy (bits)')
    if vi == 0:
        ax.set_ylabel('Human accuracy (VQA)')
    ax.set_title(f'Variant {v} (r={r:.2f}, p={p:.1e})')
    if vi == 2:
        ax.legend(fontsize=8, loc='upper right')

fig.suptitle('Human entropy vs accuracy by op × variant', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'human_entropy_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# Print per-variant correlations
for v in VARIANT_ORDER:
    vdf = qdf_all[v]
    valid = vdf[['entropy', 'accuracy']].dropna()
    r, p = stats.pearsonr(valid['entropy'], valid['accuracy'])
    print(f'Variant {v}: r={r:.3f}, p={p:.1e}')

## 6. Variant Stability: Human Entropy Across Variants

Do humans maintain similar diversity patterns across variants A/B/C?
If so, human behavior is robust to entity manipulation.

In [ ]:
# Compute per-question entropy for each variant
variant_entropy = {}
for v in VARIANT_ORDER:
    hv = human[human['variant'] == v]
    ve = hv.groupby('question_id').apply(
        answer_entropy, include_groups=False
    )
    variant_entropy[f'entropy_{v}'] = ve

ve_df = pd.DataFrame(variant_entropy).join(meta[['op', 'ent_group']])

# Pairwise correlations
print('Human entropy correlation across variants:')
for v1, v2 in [('C', 'B'), ('C', 'A'), ('B', 'A')]:
    valid = ve_df[[f'entropy_{v1}', f'entropy_{v2}']].dropna()
    r, p = stats.pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])
    print(f'  {v1} vs {v2}: r={r:.3f}, p={p:.1e}, N={len(valid)}')

print()
print('Mean entropy by variant:')
for v in VARIANT_ORDER:
    col = f'entropy_{v}'
    print(f'  {v}: {ve_df[col].mean():.3f} ± {ve_df[col].std():.3f}')

# By op within each variant
print()
print('Mean human entropy by op × variant:')
for op in OP_ORDER:
    vals = []
    for v in VARIANT_ORDER:
        col = f'entropy_{v}'
        sub = ve_df[ve_df['op'] == op]
        vals.append(f'{sub[col].mean():.2f}')
    print(f'  {op:6s}: ' + ' / '.join(f'{v}={val}' for v, val in zip(VARIANT_ORDER, vals)))

## 7. HH Agreement by Op: Do Humans Agree More on Certain Question Types?

In [ ]:
# HH SBERT by op and variant
hh_rows = []
for v in VARIANT_ORDER:
    hh_v = pc[(pc['variant'] == v) & (pc['pair_type'] == 'HH')].groupby(
        'question_id'
    )['sbert_score'].mean()
    tmp = hh_v.to_frame('hh').join(meta[['op', 'ent_group']])
    for op in OP_ORDER:
        sub = tmp[tmp['op'] == op]
        hh_rows.append({
            'Variant': v, 'Operation': OP_LABELS[op],
            'N': len(sub), 'HH SBERT': sub['hh'].mean(),
        })

hh_tbl = pd.DataFrame(hh_rows)
hh_pivot = hh_tbl.pivot(index='Operation', columns='Variant', values='HH SBERT')
hh_pivot = hh_pivot.reindex([OP_LABELS[op] for op in OP_ORDER])
hh_pivot['C-A gap'] = hh_pivot['C'] - hh_pivot['A']
display(hh_pivot.style.format('{:.3f}'))

print()
print('Key: C-A gap shows how much human agreement drops when entities are removed.')
print('Near-zero gaps mean humans are robust to entity manipulation.')

## 8. Confidence by Operation Type

Human self-reported confidence (1-5 scale, normalized to [0,1]).

In [ ]:
# Check if confidence column exists
if 'confidence' in human.columns:
    hC_conf = hC[hC['confidence'].notna()]
    conf_q = hC_conf.groupby('question_id')['confidence'].mean().rename('confidence')
    qdf_conf = qdf.join(conf_q)

    print('Human confidence by operation type (variant C):')
    for op in OP_ORDER:
        sub = qdf_conf[qdf_conf['op'] == op]
        c = sub['confidence'].dropna()
        print(f'  {op:6s}: {c.mean():.3f} ± {c.std():.3f}  (N={len(c)})')
    print(f'  {"All":6s}: {qdf_conf["confidence"].mean():.3f} ± {qdf_conf["confidence"].std():.3f}')

    # Correlation: confidence vs entropy
    valid = qdf_conf[['confidence', 'entropy']].dropna()
    r, p = stats.pearsonr(valid['confidence'], valid['entropy'])
    print(f'\nConfidence vs entropy: r={r:.3f}, p={p:.1e}')

    # Correlation: confidence vs accuracy
    valid2 = qdf_conf[['confidence', 'accuracy']].dropna()
    r2, p2 = stats.pearsonr(valid2['confidence'], valid2['accuracy'])
    print(f'Confidence vs accuracy: r={r2:.3f}, p={p2:.1e}')

    F_c, p_c = stats.f_oneway(*[
        qdf_conf[qdf_conf['op'] == op]['confidence'].dropna().values
        for op in OP_ORDER if len(qdf_conf[qdf_conf['op'] == op]['confidence'].dropna()) > 1
    ])
    print(f'\nANOVA confidence ~ op: F={F_c:.2f}, p={p_c:.4f}')
else:
    print('No confidence column in human responses.')
    # Try loading from confidence data
    conf_path = EXPORTS.parent.parent / 'evaluation/humans/confidence'
    print(f'Check: {conf_path.exists()}')

## 9. Yes/No Questions: A Special Case

Yes/no questions have constrained answer space (max entropy = 1 bit).
Do they behave differently by op?

In [ ]:
# Separate yes/no vs free-text (using variant C to identify)
hC = human[human['variant'] == 'C']
def yesno_frac(grp):
    answers = grp['response'].str.lower()
    return answers.isin(['yes', 'no']).mean()

yn_frac = hC.groupby('question_id').apply(yesno_frac, include_groups=False)
yesno_qids = set(yn_frac[yn_frac > 0.8].index)
freetext_qids = set(yn_frac[yn_frac <= 0.8].index)

print(f'Yes/no questions: {len(yesno_qids)}')
print(f'Free-text questions: {len(freetext_qids)}')

# Compare entropy by op × variant, split by yes/no vs free-text
for v in VARIANT_ORDER:
    vdf = qdf_all[v]
    print(f'\n--- Variant {v}: Mean entropy by op (Yes/No vs Free-text) ---')
    print(f'{"Op":8s} {"YN":>16s} {"Free":>16s}')
    for op in OP_ORDER:
        sub_yn = vdf[(vdf['op'] == op) & (vdf.index.isin(yesno_qids))]
        sub_ft = vdf[(vdf['op'] == op) & (vdf.index.isin(freetext_qids))]
        yn_str = f'{sub_yn["entropy"].mean():.2f} (n={len(sub_yn)})' if len(sub_yn) > 0 else '-'
        ft_str = f'{sub_ft["entropy"].mean():.2f} (n={len(sub_ft)})' if len(sub_ft) > 0 else '-'
        print(f'{op:8s} {yn_str:>16s} {ft_str:>16s}')

## 9b. Top and Bottom Questions by HH Agreement

Which questions do humans agree on most (high HH SBERT) and least (low HH SBERT)?

In [ ]:
# Top/bottom questions by HH SBERT — show all variants
def top_answers_v(qid, variant, n=3):
    hv = human[(human['variant'] == variant) & (human['question_id'] == qid)]
    counts = hv['response'].value_counts().head(n)
    return ', '.join('{} ({})'.format(a, c) for a, c in counts.items())

for v in VARIANT_ORDER:
    vdf = qdf_all[v]
    vdf_hh = vdf[vdf['hh_sbert'].notna()].copy()

    print(f'\n{"="*70}')
    print(f'VARIANT {v}: TOP 10 Highest HH SBERT (humans agree most)')
    print(f'{"="*70}\n')
    top10 = vdf_hh.nlargest(10, 'hh_sbert')
    for qid, row in top10.iterrows():
        q = str(row['question_en'])[:60]
        answers = top_answers_v(qid, v)
        print('{:6s} | {:5s} | HH={:.3f} | H={:.2f} | acc={:.3f}'.format(
            row['op'], str(row.get('ent', '')), row['hh_sbert'], row['entropy'], row['accuracy']))
        print('  Q: "{}"'.format(q))
        print('  Top answers: {}'.format(answers))
        print()

    print(f'\n{"="*70}')
    print(f'VARIANT {v}: BOTTOM 10 Lowest HH SBERT (humans disagree most)')
    print(f'{"="*70}\n')
    bot10 = vdf_hh.nsmallest(10, 'hh_sbert')
    for qid, row in bot10.iterrows():
        q = str(row['question_en'])[:60]
        answers = top_answers_v(qid, v)
        print('{:6s} | {:5s} | HH={:.3f} | H={:.2f} | acc={:.3f}'.format(
            row['op'], str(row.get('ent', '')), row['hh_sbert'], row['entropy'], row['accuracy']))
        print('  Q: "{}"'.format(q))
        print('  Top answers: {}'.format(answers))
        print()

# Op distribution comparison
print('\n--- Op distribution in top/bottom 10 by variant ---')
for v in VARIANT_ORDER:
    vdf_hh = qdf_all[v][qdf_all[v]['hh_sbert'].notna()]
    top10 = vdf_hh.nlargest(10, 'hh_sbert')
    bot10 = vdf_hh.nsmallest(10, 'hh_sbert')
    print(f'Variant {v}: Top={top10["op"].value_counts().to_dict()}, Bot={bot10["op"].value_counts().to_dict()}')

## 10. Summary

**Key findings from human-only analysis (all variants C/B/A):**

1. **Operation type drives human answer diversity** (permutation ANOVA p=0.004, Kruskal-Wallis p=0.013, eta²=0.111); entity group does not (F=0.65, p=0.63). ANOVA significance holds across all three variants.
2. **Action questions**: lowest entropy (1.79, 95% CI [0.90, 2.93]), highest accuracy (0.455) — strong linguistic priors even for humans
3. **Spatial/identity questions**: highest entropy (3.80/3.64), lowest accuracy — genuinely ambiguous without images
4. **Variant stability**: human entropy correlates r=0.89–0.92 across C/B/A — humans are robust to entity manipulation
5. **Identity questions** show the steepest entropy increase under entity removal (3.64→4.55)
6. Entropy and accuracy are strongly anticorrelated across all variants (r≈-0.77)
7. **Caveats**: small N in act (7) and ident (9) means wide bootstrap CIs; act vs count/attr contrasts are borderline
8. **Top/bottom HH agreement** patterns are consistent across variants — same questions appear at extremes regardless of entity condition